# Natężenie ruchu (Heatmap) - Historia Traffic

Ten notatnik zajmuje się tworzeniem map opóźnień ("heatmap" oraz interaktywnych punktów) w oparciu o informacje z pliku `traffic_history.csv`. Analizujemy w nim zagęszczenie i średnie opóźnienia uliczne z podziałem na godziny.

In [ ]:
import pandas as pd
import plotly.express as px
import osmnx as ox
import matplotlib.pyplot as plt

# 1. Wczytanie i przygotowanie danych samochodowych o ruchu
df = pd.read_csv("traffic_history.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.strftime('%Y-%m-%d %H:00')

# Pogrupowanie danych z perspektywy krzyżówek/ulic i godzin
df_grouped = df.groupby(['hour', 'point_name', 'lat', 'lon'], as_index=False).agg({
    'delay_sec': 'mean',
    'congestion_index_pct': 'mean',
    'current_speed_kmph': 'mean'
})

df_grouped = df_grouped.sort_values(by=['hour'])
df_grouped.head()

## Interaktywna mapa względem godziny

Wykorzystujemy `scatter_mapbox` jako formę interaktywnej "heatmapy". Animacja pokazuje zmiany w natężeniu ruchu ulicznego z biegiem czasu.

In [ ]:
fig = px.scatter_mapbox(
    df_grouped,
    lat="lat",
    lon="lon",
    color="congestion_index_pct", 
    size="delay_sec",             
    hover_name="point_name",      
    hover_data={"delay_sec": True, "congestion_index_pct": True, "current_speed_kmph": True, "lat": False, "lon": False},
    animation_frame="hour",       
    color_continuous_scale=px.colors.sequential.Inferno, 
    range_color=[0, 100],         
    zoom=11,                      
    center={"lat": 50.0647, "lon": 19.9450}, 
    title="Zmienne natężenie / opóźnienia w ruchu kołowym w Krakowie (względem godziny)",
    height=800
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

## Statyczna mapa z siatką drogową OSMnx

Naniesienie punktów bez tła dynamicznego bezprośrednio na układ dróg z biblioteki statycznej.

In [ ]:
# Pobieranie grafu drogowego dla centrum
lokalizacja = "Kraków, Poland"
G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)

# Wykreślenie siatki dróg
fig, ax = ox.plot_graph(G, node_size=0, edge_linewidth=0.5, edge_color="gray", bgcolor="white", show=False, close=False)

# Nakładanie naszych punktów historycznych (z pierwszej godziny pomiarowej)
df_first_hour = df_grouped[df_grouped['hour'] == df_grouped['hour'].iloc[0]]

scatter = ax.scatter(
    df_first_hour['lon'], 
    df_first_hour['lat'], 
    c=df_first_hour['congestion_index_pct'], 
    cmap='inferno',
    s=df_first_hour['delay_sec'], 
    alpha=0.8,
    zorder=3
)
plt.colorbar(scatter, ax=ax, label="Congestion Index (%)")
plt.title(f"Siatka ulic OSM + wskaźnik natężenia dla {df_first_hour['hour'].iloc[0]}")
plt.show()